### 기본 라이브러리

In [17]:
# 기본
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 경고 뜨지 않게 설정
import warnings
warnings.filterwarnings('ignore')

# 그래프 설정
sns.set()

# 그래프 기본 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 알고리즘
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# 학습용과 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split

# 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

# 평가함수
# 분류용
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import make_scorer

# 회귀용
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

# 모델의 최적의 하이퍼 파라미터를 찾기 위한 도구
from sklearn.model_selection import GridSearchCV

# 머신러닝 알고리즘 - 분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

### [1단계] 이진 분류

1. E vs E가 아닌 것

In [6]:
# 데이터 불러오기
df = pd.read_csv("./result/상위20개컬럼.csv")

In [18]:
# 이진 타겟 생성
df["is_E"] = (df["Segment"] == "E").astype(int)

# 입력 변수와 타겟 분리
X = df.drop(columns=["Segment", "is_E"])
y = df["is_E"]

# 입력 데이터 표준화
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 훈련/검증 나누기
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)

In [19]:
# 모델 정의 및 학습
model_bin = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    use_label_encoder=False,
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    tree_method="hist", 
    device="cuda"
)
model_bin.fit(X_train, y_train)

# 예측 및 결과 확인
y_pred = model_bin.predict(X_val)
print("[1단계] E vs Not E 이진 분류 결과")
print(confusion_matrix(y_val, y_pred))
print(classification_report(y_val, y_pred))

[1단계] E vs Not E 이진 분류 결과
[[ 65969  29621]
 [ 19184 365226]]
              precision    recall  f1-score   support

           0       0.77      0.69      0.73     95590
           1       0.92      0.95      0.94    384410

    accuracy                           0.90    480000
   macro avg       0.85      0.82      0.83    480000
weighted avg       0.90      0.90      0.90    480000



#### 1단계 결과 해석
| 항목                   | 해석                                              |
| -------------------- | ----------------------------------------------- |
| **Accuracy: 90%**    | 전체 예측 정확도 우수                                    |
| **Class 1 (E)**      | Precision 0.92 / Recall 0.95 → **잘 예측됨**        |
| **Class 0 (비E)**     | Recall 0.69 → **E 아닌 걸 놓치는 경우 있음**, 하지만 큰 문제 아님 |
| **Macro avg = 0.83** | 클래스 간 균형도 준수                                    |  

- 요약: E 클래스는 잘 잡아내고 있고, 이제 비E 데이터만 추출해서 A/B/C/D 분류로 넘어간다.


### [2단계] Not E 데이터만 추출 -> A/B/C/d 분류

In [20]:
# 1. 비E 데이터만 추출
df_not_e = df[df['Segment'] != 'E'].copy()

# 2. 입력 변수, 타겟 재정의
X_not_e = df_not_e.drop(columns=['Segment', 'is_E'])
y_not_e = df_not_e['Segment']

# 3. 타겟 인코딩
le_abcd = LabelEncoder()
y_not_e_encoded = le_abcd.fit_transform(y_not_e)

# 4. 학습/검증 나누기
X_train_ne, X_val_ne, y_train_ne, y_val_ne = train_test_split(
    X_not_e, y_not_e_encoded, test_size=0.2, stratify=y_not_e_encoded, random_state=42
)

In [21]:
# 5. XGBoost 다중분류 모델 정의 및 학습
model_abcd = XGBClassifier(
    objective='multi:softmax',
    num_class=4,
    eval_metric='mlogloss',
    use_label_encoder=False,
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    tree_method="hist",
    device="cuda"
)

model_abcd.fit(X_train_ne, y_train_ne)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device='cuda', early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None, num_class=4, ...)

In [22]:
# 6. 결과 확인
y_pred_ne = model_abcd.predict(X_val_ne)
print("[2단계] Not E 데이터 A/B/C/D 분류 결과")
print(classification_report(y_val_ne, y_pred_ne, target_names=le_abcd.classes_))

[2단계] Not E 데이터 A/B/C/D 분류 결과
              precision    recall  f1-score   support

           A       0.74      0.19      0.30       194
           B       1.00      0.03      0.07        29
           C       0.73      0.52      0.61     25518
           D       0.84      0.93      0.88     69849

    accuracy                           0.82     95590
   macro avg       0.83      0.42      0.47     95590
weighted avg       0.81      0.82      0.81     95590



#### 2단계 결과 해석
| 클래스   | Precision | Recall | F1-score | 샘플 수   | 해석                        |
| ----- | --------- | ------ | -------- | ------ | ------------------------- |
| **A** | 0.74      | 0.19   | 0.30     | 194    | 거의 다 놓침 (Recall 낮음)       |
| **B** | 1.00      | 0.03   | 0.07     | 29     | 너무 적게 잡음 (고정도지만 감지 거의 안됨) |
| **C** | 0.73      | 0.52   | 0.61     | 25,518 | 괜찮음                       |
| **D** | 0.84      | 0.93   | 0.88     | 69,849 | 매우 우수                     |

#### 2단계 kfold 교차검증

In [ ]:
# 평가 지표 정의 
f1_macro = make_scorer(f1_score, average='macro')

# 교차검증 설정
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# 모델 정의
model = XGBClassifier(
    objective='multi:softmax',
    num_class=4,
    eval_metric='mlogloss',
    use_label_encoder=False,
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    tree_method='hist',
    device='cuda'
)

# 교차검증 실행 (X_not_e, y_not_e_encoded는 사전 정의된 데이터 사용)
cv_scores = cross_val_score(model, X_not_e, y_not_e_encoded, cv=cv, scoring=f1_macro)


In [28]:
# 결과 출력
print("[Fold별 f1_macro score]")
for i, score in enumerate(cv_scores, 1):
    print(f"Fold {i}: {score:.4f}")
print(f"평균 f1_macro score: {cv_scores.mean():.4f}")

[Fold별 f1_macro score]
Fold 1: 0.4305
Fold 2: 0.5239
Fold 3: 0.4335
Fold 4: 0.4258
Fold 5: 0.4555
Fold 6: 0.4894
Fold 7: 0.4350
Fold 8: 0.5033
Fold 9: 0.4501
Fold 10: 0.4537
평균 f1_macro score: 0.4601


다음 단계에서 진행할 대략적인 흐름

- A/B 샘플만 추출
- 피처 분포 시각화 + 통계 비교 (sns.histplot, df.groupby().mean())
- 별도 A/B 분류 모델 만들어서 교차검증
- 클래스 분리 잘 되는지 확인 → 성능 낮으면 피처 엔지니어링 or 오버샘플링